# Skin Cancer Detection Dataset — EDA + CNN

This notebook is designed for Kaggle. It explores image folders, class balance, image dimensions, sample images, and then trains a CNN image classifier.

**Important medical limitation:** This is an educational machine-learning exercise, not a diagnostic tool. Predictions must not be used to diagnose or rule out skin cancer. A qualified healthcare professional must assess concerning skin changes.

## 1. Imports and reproducibility

In [ ]:
import os, glob, random, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True
sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    tf.random.set_seed(SEED)
    print("TensorFlow:", tf.__version__)
except ImportError as exc:
    raise ImportError("TensorFlow is required. Kaggle usually provides it; enable a TensorFlow-compatible environment.") from exc

## 2. Locate the dataset

In [ ]:
# Kaggle datasets usually appear under /kaggle/input/<dataset-folder>.
INPUT_ROOT = Path("/kaggle/input")
if not INPUT_ROOT.exists():
    INPUT_ROOT = Path(".")

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
all_files = [p for p in INPUT_ROOT.rglob("*") if p.is_file()] if INPUT_ROOT.exists() else []
image_files = [p for p in all_files if p.suffix.lower() in image_extensions]

print("Input root:", INPUT_ROOT.resolve())
print("Image files discovered:", len(image_files))
print("CSV files discovered:", [str(p) for p in all_files if p.suffix.lower() == ".csv"][:30])

if not image_files:
    raise FileNotFoundError(
        "No images were found. In Kaggle, click Add Input and attach the skin-cancer-detection dataset, "
        "then run this cell again."
    )

# Show a small folder overview.
for p in sorted({x.parent for x in image_files})[:30]:
    print(p, "->", sum(1 for x in image_files if x.parent == p), "images")

## 3. Build image manifest and infer labels from folders

In [ ]:
# This works when images are arranged in class subfolders, for example:
# dataset/melanoma/image1.jpg, dataset/benign/image2.jpg
# If the dataset has a different structure, inspect the printed folder overview above and adapt label extraction.

records = []
for path in image_files:
    relative_parts = path.relative_to(INPUT_ROOT).parts if str(path).startswith(str(INPUT_ROOT)) else path.parts
    # The immediate parent folder is treated as the class label.
    label = path.parent.name
    records.append({
        "path": str(path),
        "label": label,
        "extension": path.suffix.lower(),
        "parent_folder": str(path.parent)
    })

manifest = pd.DataFrame(records)
print("Manifest shape:", manifest.shape)
print("Unique inferred folder labels:", manifest["label"].nunique())
display(manifest.head())

label_counts = manifest["label"].value_counts()
display(label_counts.rename_axis("class").to_frame("image_count"))

plt.figure(figsize=(max(8, len(label_counts) * 0.8), 5))
sns.barplot(x=label_counts.index, y=label_counts.values)
plt.title("Image Count per Inferred Class Folder")
plt.xlabel("Folder label")
plt.ylabel("Number of images")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Inspect image properties and sample images

In [ ]:
# Inspect a sample of images for dimensions and readability.
sample_paths = manifest["path"].sample(min(500, len(manifest)), random_state=SEED).tolist()
image_info = []
for path in sample_paths:
    try:
        with Image.open(path) as im:
            image_info.append({
                "path": path,
                "width": im.width,
                "height": im.height,
                "mode": im.mode,
                "format": im.format
            })
    except Exception as e:
        image_info.append({"path": path, "width": np.nan, "height": np.nan, "mode": "UNREADABLE", "format": None})

image_info = pd.DataFrame(image_info)
print("Sampled image properties:")
display(image_info.head())
print("Unreadable sampled files:", int(image_info["width"].isna().sum()))

valid_dims = image_info.dropna(subset=["width", "height"])
if not valid_dims.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(valid_dims["width"], bins=25, ax=axes[0])
    axes[0].set_title("Image Width (sample)")
    sns.histplot(valid_dims["height"], bins=25, ax=axes[1])
    axes[1].set_title("Image Height (sample)")
    plt.tight_layout()
    plt.show()

# Display up to 3 images per class.
classes = sorted(manifest["label"].unique())
ncols = min(4, max(1, len(classes)))
nrows = min(3, len(classes))
fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.5*nrows), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for i, label in enumerate(classes[:nrows*ncols]):
    paths = manifest.loc[manifest["label"] == label, "path"].sample(
        min(1, (manifest["label"] == label).sum()), random_state=SEED
    ).tolist()
    if paths:
        try:
            with Image.open(paths[0]) as im:
                axes.flat[i].imshow(im.convert("RGB"))
                axes.flat[i].set_title(label)
                axes.flat[i].axis("off")
        except Exception:
            axes.flat[i].set_title(label + " (unreadable)")
plt.tight_layout()
plt.show()

## 5. Check duplicates and class imbalance

In [ ]:
print("Exact duplicate file paths:", int(manifest["path"].duplicated().sum()))
print("Unique paths:", manifest["path"].nunique())

# Check duplicate file contents by file size + SHA256 on a bounded sample.
import hashlib
def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

hash_records = []
for path in manifest["path"].tolist():
    try:
        hash_records.append((path, os.path.getsize(path), sha256_file(path)))
    except Exception:
        pass
hash_df = pd.DataFrame(hash_records, columns=["path", "size_bytes", "sha256"])
print("Duplicate content groups (same hash, excluding singletons):",
      int((hash_df.groupby("sha256").size() > 1).sum()) if not hash_df.empty else 0)

print("\\nClass proportions:")
display((label_counts / label_counts.sum()).rename("proportion").to_frame())

## 6. Prepare train, validation, and test sets

In [ ]:
from sklearn.model_selection import train_test_split

# Drop files that PIL cannot read.
readable_paths = []
for path in manifest["path"]:
    try:
        with Image.open(path) as im:
            im.verify()
        readable_paths.append(path)
    except Exception:
        pass

data = manifest[manifest["path"].isin(readable_paths)].copy()
data["label"] = data["label"].astype(str)
counts = data["label"].value_counts()
data = data[data["label"].isin(counts[counts >= 3].index)].copy()

if data["label"].nunique() < 2:
    raise ValueError(
        "At least two class folders with 3 or more readable images are required. "
        "The inferred folder labels may be incorrect; inspect the dataset structure and adjust label extraction."
    )

# Split each class approximately 70/15/15 with stratification.
train_df, temp_df = train_test_split(
    data, test_size=0.30, random_state=SEED, stratify=data["label"]
)
valid_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)

print("Train:", train_df.shape, "Validation:", valid_df.shape, "Test:", test_df.shape)
print("Train class counts:")
display(train_df["label"].value_counts().to_frame("images"))

## 7. Create TensorFlow image pipelines

In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

class_names = sorted(data["label"].unique())
class_to_id = {name: i for i, name in enumerate(class_names)}
print("Class mapping:", class_to_id)

def make_dataset(frame, training=False):
    paths = frame["path"].to_numpy()
    labels = frame["label"].map(class_to_id).to_numpy(dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load_image(path, label):
        image_bytes = tf.io.read_file(path)
        image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(min(len(frame), 2000), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
valid_ds = make_dataset(valid_df)
test_ds = make_dataset(test_df)

## 8. Implement CNN

In [ ]:
num_classes = len(class_names)

cnn = models.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.BatchNormalization(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.BatchNormalization(),
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.35),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.25),
    layers.Dense(num_classes, activation="softmax")
])

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
cnn.summary()

## 9. Train with early stopping

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6)
]

history = cnn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

## 10. Plot learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="Train")
axes[0].plot(history.history["val_loss"], label="Validation")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Train")
axes[1].plot(history.history["val_accuracy"], label="Validation")
axes[1].set_title("Accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()

## 11. Evaluate on held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

test_loss, test_accuracy = cnn.evaluate(test_ds, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

y_true = np.concatenate([y.numpy() for _, y in test_ds])
probs = cnn.predict(test_ds, verbose=0)
y_pred = np.argmax(probs, axis=1)

print("\\nClassification report:")
print(classification_report(
    y_true, y_pred, labels=list(range(len(class_names))),
    target_names=class_names, zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(max(7, len(class_names)), max(6, len(class_names)*0.8)))
disp.plot(ax=ax, xticks_rotation=45, values_format="d", colorbar=False)
plt.title("CNN Confusion Matrix")
plt.tight_layout()
plt.show()

## 12. Save model and dataset manifest

In [ ]:
cnn.save("skin_cancer_cnn.keras")
manifest.to_csv("skin_cancer_image_manifest.csv", index=False)
print("Saved model: skin_cancer_cnn.keras")
print("Saved manifest: skin_cancer_image_manifest.csv")

## Conclusion and limitations

- EDA includes image counts by inferred class folder, sample images, image dimensions, unreadable files, class proportions, and duplicate-file checks.
- The CNN classifies images according to the folder labels found in the dataset.
- The notebook does not assume the dataset has a particular number of classes.
- If the dataset stores labels in a CSV rather than class-named folders, modify the manifest-building cell to join image paths with that CSV.
- This model is for learning only and must not be used for diagnosis, treatment decisions, or reassurance that a lesion is harmless. Real-world use requires clinical validation and qualified medical assessment.